<a href="https://colab.research.google.com/github/abay-qkt/mac-kindle-sqlite-analysis/blob/main/Mac%E7%89%88Kindle_ZBOOK%E8%A7%A3%E6%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ読み込み

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

import plotly.express as px

from pathlib import Path
import plistlib
from typing import Any, Dict

def resolve_ns_keyed_archive_fully(data: bytes) -> Any:
    if pd.isna(data):
        return np.nan
    root = plistlib.loads(data)
    objects = root["$objects"]
    top_uid = root["$top"]["root"]

    # クラスID → クラス名
    class_map = {
        idx: obj["$classname"]
        for idx, obj in enumerate(objects)
        if isinstance(obj, dict) and "$classname" in obj
    }

    def resolve(obj: Any, memo: Dict[int, Any]) -> Any:
        if isinstance(obj, plistlib.UID):
            idx = obj.data
            if idx in memo:
                return memo[idx]
            raw = objects[idx]
            resolved = resolve(raw, memo)
            memo[idx] = resolved
            return resolved

        elif isinstance(obj, list):
            return [resolve(item, memo) for item in obj]

        elif isinstance(obj, dict):
            # クラスID に基づいて判定
            class_id = obj.get("$class")
            class_name = class_map.get(class_id.data) if isinstance(class_id, plistlib.UID) else None

            # NSMutableArray / NSArray の展開
            if class_name in ("NSMutableArray", "NSArray") and "NS.objects" in obj:
                return resolve(obj["NS.objects"], memo)

            # NSMutableDictionary / NSDictionary の展開
            if class_name in ("NSMutableDictionary", "NSDictionary") and "NS.keys" in obj and "NS.objects" in obj:
                keys = resolve(obj["NS.keys"], memo)
                vals = resolve(obj["NS.objects"], memo)
                return dict(zip(keys, vals))

            # 通常の辞書展開
            return {
                resolve(k, memo): resolve(v, memo)
                for k, v in obj.items()
                if not (isinstance(k, str) and k.startswith("$"))
            }

        else:
            return obj

    return resolve(top_uid, {})

def get_author(x):
    if not isinstance(x, dict):
        return np.nan
    attributes = x.get("attributes")
    if not isinstance(attributes, dict):
        return np.nan
    authors = attributes.get("authors")
    if not isinstance(authors, dict):
        return np.nan
    author = authors.get("author")
    if author is None:
        return np.nan
    if isinstance(author, list):  # 複数人の場合リストで格納されている
        return "/".join(author)  # /で結合した文字列で返す
    return author  # 一人の場合単なる文字列なのでそのまま返す

def get_origin_type(x):
    if not isinstance(x, dict):
        return np.nan
    attributes = x.get("attributes")
    if not isinstance(attributes, dict):
        return np.nan
    origins = attributes.get("origins")
    if not isinstance(origins, dict):
        return np.nan
    origin = origins.get("origin")
    if not isinstance(origin, dict):
        return np.nan
    type_value = origin.get("type")
    if type_value is None:
        return np.nan
    return type_value

def get_date(x,key):
    if not isinstance(x, dict):
        return np.nan
    attributes = x.get("attributes")
    if not isinstance(attributes, dict):
        return np.nan
    date = attributes.get(key)
    return date

conn = sqlite3.connect("BookData.sqlite")
book_df = pd.read_sql_query('SELECT * FROM ZBOOK', conn)

book_df["ASIN"]=book_df["ZBOOKID"].map(lambda x:x.split(":")[1].split("-")[0])
book_df["ZSYNCMETADATAATTRIBUTES"] = book_df["ZSYNCMETADATAATTRIBUTES"].map(resolve_ns_keyed_archive_fully)

book_df["title_pron"] = book_df["ZSORTTITLE"]
book_df["title"] = book_df["ZDISPLAYTITLE"]

book_df["authors"] = book_df["ZSYNCMETADATAATTRIBUTES"].map(get_author)
book_df["publishers"] = book_df["ZRAWPUBLISHER"]

book_df["origin_type"] = book_df["ZSYNCMETADATAATTRIBUTES"].map(get_origin_type)

book_df = book_df[book_df["origin_type"]!="Sample"] # 書籍サンプルを除外
book_df = book_df[book_df["origin_type"]!="KindleDictionary"] # デフォルトで入っている辞書を除外

# 出版日と購入日時を取得
book_df["publication_date"] = book_df["ZSYNCMETADATAATTRIBUTES"].map(lambda x:get_date(x,"publication_date"))
book_df["purchase_date"] = book_df["ZSYNCMETADATAATTRIBUTES"].map(lambda x:get_date(x,"purchase_date"))
# datetime型に変換
book_df["publication_date"] = pd.to_datetime(book_df["publication_date"]).dt.tz_localize(None) # タイムゾーン情報無くす
book_df["purchase_date"] = pd.to_datetime(book_df["purchase_date"]).dt.tz_convert('Asia/Tokyo').dt.tz_localize(None) # JSTにしてからタイムゾーン情報無くす

book_df = book_df.dropna(subset=["purchase_date"])

# データ確認

In [ ]:
book_df[["title","authors","publishers","origin_type","publication_date","purchase_date"]].head(5)

,title,authors,publishers,origin_type,publication_date,purchase_date
42,逃げ上手の若君 6 (ジャンプコミックスDIGITAL),松井優征,集英社,Purchase,2022-06-03,2025-03-31 20:48:04
43,逃げ上手の若君 19 (ジャンプコミックスDIGITAL),松井優征,集英社,Purchase,2025-02-04,2025-03-31 20:48:04
44,逃げ上手の若君 2 (ジャンプコミックスDIGITAL),松井優征,集英社,Purchase,2021-08-04,2025-03-31 20:48:03
45,逃げ上手の若君 1 (ジャンプコミックスDIGITAL),松井優征,集英社,Purchase,2021-07-02,2025-03-31 20:48:04
46,逃げ上手の若君 17 (ジャンプコミックスDIGITAL),松井優征,集英社,Purchase,2024-09-04,2025-03-31 20:48:04


In [ ]:
# 購入冊数
book_df.shape[0]

3361

In [ ]:
# 年ごとの購入冊数
px.bar(
    book_df
    .groupby(pd.Grouper(key='purchase_date',freq='YS'),as_index=False)
    .size(),
    x='purchase_date',y='size'
)

In [ ]:
# 月ごとの購入冊数
px.bar(
    book_df
    .groupby(pd.Grouper(key='purchase_date',freq='MS'),as_index=False)
    .size(),
    x='purchase_date',y='size'
)

In [ ]:
# タグごとの購入冊数
book_df["ZCONTENTTAGS"].value_counts()

,count
ZCONTENTTAGS,
;MANGA,3048
,220
;MANGA;COMICS,93


In [ ]:
# 著者ごとの購入冊数（共著は別人扱い）
book_df["authors"].value_counts().head(5)

,count
authors,
雷句誠,66
荒川弘,63
石井さだよし/星野茂樹,50
石田スイ,45
青樹佑夜/綾峰欄人,42


In [ ]:
# 著者ごとの購入冊数
book_df["authors"].str.split("/").explode().value_counts().head(5)

,count
authors,
荒川弘,86
雷句誠,66
石井さだよし,50
星野茂樹,50
諫山創,46


In [ ]:
# 出版社ごとの購入冊数
book_df["publishers"].value_counts().head(5)

,count
publishers,
講談社,1139
KADOKAWA,572
集英社,370
小学館,142
,104
